In [100]:
pip install tensorboard

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [101]:
import pandas as pd
from datetime import datetime, timedelta
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.logger import configure
import seaborn as sns
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Set the seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Define paths
file_path = "C:/Users/nikhi/Desktop/Capstone project/historical_data.csv"
MODEL_DIR = "models"
LOG_DIR = "logs"
RESULTS_DIR = "results"

# Create directories if they don't exist
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [102]:
def prepare_stock_data(file_path):
    """
    Prepare stock data from Excel/CSV file for risk analysis

    Args:
        file_path (str): Path to the Excel/CSV file

    Returns:
        dict: Dictionary of DataFrames with processed data for each stock
    """
    # Determine file type and read accordingly
    if file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        df = pd.read_excel(file_path)
    else:  # Assume CSV
        df = pd.read_csv(file_path)

    print(f"Original data shape: {df.shape}")

    # Convert date to datetime if not already
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'])

    # Sort by ticker and date
    df = df.sort_values(['tic', 'date'])

    # Calculate additional features for risk assessment
    stock_dfs = {}
    skipped_stocks = 0

    # Process each stock separately
    for ticker in df['tic'].unique():
        stock_data = df[df['tic'] == ticker].copy()

        # Ensure we have enough data (at least 1 year)
        if len(stock_data) < 252:  # Approx trading days in a year
            print(f"Skipping {ticker} - insufficient data ({len(stock_data)} rows)")
            skipped_stocks += 1
            continue

        # Calculate daily returns
        stock_data['daily_return'] = stock_data['close'].pct_change()

        # Calculate volatility (20-day and 60-day)
        stock_data['volatility_20d'] = stock_data['daily_return'].rolling(window=20).std() * np.sqrt(252)
        stock_data['volatility_60d'] = stock_data['daily_return'].rolling(window=60).std() * np.sqrt(252)

        # Calculate drawdowns
        stock_data['cumulative_return'] = (1 + stock_data['daily_return']).cumprod()
        stock_data['rolling_max'] = stock_data['cumulative_return'].cummax()
        stock_data['drawdown'] = (stock_data['cumulative_return'] / stock_data['rolling_max']) - 1
        stock_data['max_drawdown_60d'] = stock_data['drawdown'].rolling(window=60).min()

        # Calculate trading volume metrics
        stock_data['volume_change'] = stock_data['volume'].pct_change()
        stock_data['volume_ma10'] = stock_data['volume'].rolling(window=10).mean()
        stock_data['relative_volume'] = stock_data['volume'] / stock_data['volume_ma10']

        # Additional features for SAC model

        # Calculate price momentum features
        stock_data['price_ma5'] = stock_data['close'].rolling(window=5).mean()
        stock_data['price_ma20'] = stock_data['close'].rolling(window=20).mean()
        stock_data['price_ma50'] = stock_data['close'].rolling(window=50).mean()

        # Price momentum indicators
        stock_data['momentum_5d'] = stock_data['close'] / stock_data['price_ma5'] - 1
        stock_data['momentum_20d'] = stock_data['close'] / stock_data['price_ma20'] - 1
        stock_data['momentum_50d'] = stock_data['close'] / stock_data['price_ma50'] - 1

        # Calculate rate of change
        stock_data['roc_5d'] = stock_data['close'].pct_change(periods=5)
        stock_data['roc_20d'] = stock_data['close'].pct_change(periods=20)

        # Calculate weighted return metric (more recent returns have higher weight)
        weights = np.array([0.6, 0.3, 0.1])  # Weights for 1-day, 5-day, and 20-day returns
        # Handle NaN values in component returns
        daily_return = stock_data['daily_return'].fillna(0)
        roc_5d = stock_data['roc_5d'].fillna(0)
        roc_20d = stock_data['roc_20d'].fillna(0)

        stock_data['weighted_return'] = (
            weights[0] * daily_return +
            weights[1] * roc_5d +
            weights[2] * roc_20d
        )

        # Calculate relative strength index (RSI)
        delta = stock_data['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        # Handle division by zero in RSI calculation
        rs = gain / loss.replace(0, np.nan)
        rs = rs.fillna(0)  # If loss is 0, set rs to 0
        stock_data['rsi_14d'] = 100 - (100 / (1 + rs))
        # Ensure RSI is within bounds
        stock_data['rsi_14d'] = stock_data['rsi_14d'].clip(0, 100)

        # Drop NaN values that result from calculations
        stock_data = stock_data.dropna()

        # Only include stock if we still have meaningful data after calculations
        if len(stock_data) >= 126:  # At least ~6 months of data
            stock_dfs[ticker] = stock_data
        else:
            skipped_stocks += 1

    print(f"Processed {len(stock_dfs)} stocks, skipped {skipped_stocks} stocks")
    return stock_dfs

def calculate_risk_metrics(stock_dfs, lookback_period=252):
    """
    Calculate risk metrics for each stock

    Args:
        stock_dfs (dict): Dictionary of stock DataFrames
        lookback_period (int): Period to use for calculations (default 1 year)

    Returns:
        DataFrame: Summary of risk metrics for all stocks
    """
    risk_metrics = []

    for ticker, data in stock_dfs.items():
        # Use recent data only
        recent_data = data.iloc[-lookback_period:] if len(data) > lookback_period else data

        # Skip stocks with zero or constant prices (no price movement)
        if recent_data['daily_return'].std() == 0 or recent_data['close'].nunique() <= 1:
            print(f"Skipping {ticker} - no price variation")
            continue

        # Calculate key risk metrics
        metrics = {
            'ticker': ticker,
            'avg_daily_return': recent_data['daily_return'].mean(),
            'annualized_return': recent_data['daily_return'].mean() * 252,
            'volatility': recent_data['daily_return'].std() * np.sqrt(252),
            'max_drawdown': recent_data['drawdown'].min(),
            'sharpe_ratio': (recent_data['daily_return'].mean() * 252) /
                           (recent_data['daily_return'].std() * np.sqrt(252))
                           if recent_data['daily_return'].std() > 0 else 0,
            'avg_volume': recent_data['volume'].mean(),
            'volume_volatility': recent_data['volume_change'].std() if not pd.isna(recent_data['volume_change'].std()) else 0,

            # Additional metrics for SAC model
            'avg_rsi': recent_data['rsi_14d'].mean(),
            'rsi_volatility': recent_data['rsi_14d'].std(),
            'avg_momentum_20d': recent_data['momentum_20d'].mean(),
            'weighted_return_avg': recent_data['weighted_return'].mean(),
            'weighted_return_vol': recent_data['weighted_return'].std(),
            'data_points': len(recent_data)
        }

        risk_metrics.append(metrics)

    # Convert to DataFrame
    if not risk_metrics:
        print("No valid stocks with risk metrics found!")
        return pd.DataFrame()

    risk_df = pd.DataFrame(risk_metrics)

    # Handle extreme values and missing data
    numeric_columns = risk_df.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        # Replace infinity values
        risk_df[col] = risk_df[col].replace([np.inf, -np.inf], np.nan)

        # Handle missing values with median imputation
        if risk_df[col].isna().any():
            median_val = risk_df[col].median()
            risk_df[col] = risk_df[col].fillna(median_val)

    # Calculate risk score components with expanded metrics
    vol_component = (risk_df['volatility'] / risk_df['volatility'].max()) if risk_df['volatility'].max() > 0 else 0
    dd_component = (risk_df['max_drawdown'].abs() / risk_df['max_drawdown'].abs().max()) if risk_df['max_drawdown'].abs().max() > 0 else 0
    vol_vol_component = (risk_df['volume_volatility'] / risk_df['volume_volatility'].max()) if risk_df['volume_volatility'].max() > 0 else 0

    # Sharpe ratio is inversely related to risk (higher is better)
    sharpe_range = risk_df['sharpe_ratio'].max() - risk_df['sharpe_ratio'].min()
    if sharpe_range > 0:
        sharpe_component = 1 - ((risk_df['sharpe_ratio'] - risk_df['sharpe_ratio'].min()) / sharpe_range)
    else:
        sharpe_component = 0.5  # Neutral value if all stocks have the same Sharpe ratio

    # RSI component (extreme values indicate higher risk)
    rsi_deviation = abs(risk_df['avg_rsi'] - 50) / 50  # Deviation from neutral RSI (50)
    rsi_component = rsi_deviation / rsi_deviation.max() if rsi_deviation.max() > 0 else 0

    # Momentum component (high momentum can indicate higher risk)
    momentum_component = abs(risk_df['avg_momentum_20d'])
    momentum_component = momentum_component / momentum_component.max() if momentum_component.max() > 0 else 0

    # Add weighted risk score calculation with expanded components
    risk_df['risk_score'] = (
        vol_component * 0.3 +
        dd_component * 0.25 +
        vol_vol_component * 0.1 +
        sharpe_component * 0.15 +
        rsi_component * 0.1 +
        momentum_component * 0.1
    )

    # Ensure all stocks have a valid risk score
    if risk_df['risk_score'].isna().any():
        print(f"Warning: {risk_df['risk_score'].isna().sum()} stocks have NaN risk scores")
        # Assign median risk score to stocks with NaN
        risk_df['risk_score'] = risk_df['risk_score'].fillna(risk_df['risk_score'].median())

    # Ensure we don't have exactly 0 or 1 risk scores (adjust slightly if needed)
    risk_df.loc[risk_df['risk_score'] == 0, 'risk_score'] = 0.001
    risk_df.loc[risk_df['risk_score'] == 1, 'risk_score'] = 0.999

    # Classify risk levels using qcut for equal-sized groups
    try:
        risk_df['risk_category'] = pd.qcut(
            risk_df['risk_score'],
            q=3,
            labels=['Low', 'Medium', 'High']
        )
    except ValueError:
        # Fall back to manual classification if qcut fails
        risk_thresholds = [0, 0.33, 0.67, 1.0]
        risk_df['risk_category'] = pd.cut(
            risk_df['risk_score'],
            bins=risk_thresholds,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

    return risk_df

def save_risk_analysis(risk_df, filename='risk_analysis.csv'):
    """
    Save risk analysis results to a CSV file
    
    Args:
        risk_df: DataFrame with risk classifications
        filename: Output file name
    """
    # Save to CSV
    risk_df.to_csv(filename, index=False)
    print(f"Risk analysis saved to {filename}")

In [103]:
class StockRiskEnv(gym.Env):
    """
    Custom Gym environment for stock risk assessment using RL.
    This environment simulates portfolio management with risk constraints.
    Compatible with Stable Baselines 3 using Gymnasium.
    """
    
    metadata = {"render_modes": ["human"]}
    
    def __init__(self, stock_data_dict, initial_investment=10000.0, max_steps=252, window_size=20, min_common_dates=60, render_mode=None):
        super(StockRiskEnv, self).__init__()
        
        self.stock_data_dict = stock_data_dict
        self.tickers = list(stock_data_dict.keys())
        self.num_stocks = len(self.tickers)
        self.initial_investment = initial_investment
        self.max_steps = max_steps
        self.window_size = window_size
        self.min_common_dates = min_common_dates
        self.render_mode = render_mode
        
        # Current step and episode data
        self.current_step = 0
        self.portfolio_value = initial_investment
        self.portfolio_history = []
        self.risk_adjusted_returns = []
        
        # Prepare observation and action spaces
        # State space: For each stock: normalized_price, volatility, momentum, rsi, volume_ratio
        # Plus portfolio state: current_allocation per stock, portfolio_return, portfolio_volatility
        state_dim = (self.num_stocks * 5) + (self.num_stocks + 2)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(state_dim,), dtype=np.float32
        )
        
        # Action space: portfolio weights for each stock (continuous between 0 and 1)
        # The weights will be normalized to sum to 1
        self.action_space = spaces.Box(
            low=0, high=1, shape=(self.num_stocks,), dtype=np.float32
        )
        
        # Prepare data features and select common date range
        self._prepare_data()
        
        # Allocations (portfolio weights) - INITIALIZE AS ARRAY!
        self.allocations = np.ones(self.num_stocks) / self.num_stocks
        
        # Risk_free_rate (for Sharpe ratio calculation)
        self.risk_free_rate = 0.02 / 252  # Daily risk-free rate (approximately 2% annual)
        
    def _prepare_data(self):
        """Prepare and align data for all stocks to a common date range"""
        print(f"Preparing environment with {len(self.stock_data_dict)} stocks")
        
        # Check if 'date' column is present in all dataframes
        for ticker, df in self.stock_data_dict.items():
            if 'date' not in df.columns:
                print(f"Warning: 'date' column not found in data for {ticker}. Adding date column.")
                # Create a date column as index
                df['date'] = pd.date_range(start='2020-01-01', periods=len(df))
        
        # Find common date range for all stocks
        common_dates = None
        for ticker, df in self.stock_data_dict.items():
            # Debug: see what dates are available
            print(f"Stock {ticker} has {len(df)} rows with date range: {df['date'].min()} to {df['date'].max() if len(df) > 0 else 'N/A'}")
            dates = set(df['date'])
            if common_dates is None:
                common_dates = dates
            else:
                common_dates = common_dates.intersection(dates)
        
        if not common_dates:
            print("No common dates found across all stocks. Using most recent dates from each stock.")
            # Alternative approach: Instead of requiring perfect overlap,
            # take the most recent valid data for each stock
            self._prepare_data_most_recent()
            return
            
        common_dates = sorted(list(common_dates))
        print(f"Found {len(common_dates)} common dates across all stocks.")
        
        if len(common_dates) < self.min_common_dates:
            print(f"Warning: Only {len(common_dates)} common dates found, which is below the minimum of {self.min_common_dates}.")
            print("Using most recent dates from each stock instead.")
            self._prepare_data_most_recent()
            return
            
        # Adjust max_steps if needed
        if len(common_dates) < self.max_steps + self.window_size:
            old_max_steps = self.max_steps
            self.max_steps = len(common_dates) - self.window_size - 1
            if self.max_steps < 10:  # Minimum viable episode length
                print(f"Warning: Adjusted max_steps would be {self.max_steps}, which is too small.")
                print("Using most recent dates from each stock instead.")
                self._prepare_data_most_recent()
                return
            print(f"Adjusting max_steps from {old_max_steps} to {self.max_steps} based on available common dates.")
        
        # Trim data to common dates
        self.aligned_data = {}
        for ticker, df in self.stock_data_dict.items():
            # Filter to common dates and sort
            filtered_df = df[df['date'].isin(common_dates)].sort_values('date')
            # Select only needed columns
            selected_cols = ['date', 'close', 'daily_return', 'volatility_20d', 
                             'momentum_20d', 'rsi_14d', 'relative_volume']
            
            # Ensure all needed columns exist
            for col in selected_cols:
                if col not in filtered_df.columns:
                    print(f"Warning: Column {col} not found in data for {ticker}. Using default values.")
                    if col == 'close':
                        filtered_df[col] = 100.0  # Default price
                    elif col == 'daily_return':
                        filtered_df[col] = 0.0  # Default return
                    elif col == 'volatility_20d':
                        filtered_df[col] = 0.2  # Default volatility
                    elif col == 'momentum_20d':
                        filtered_df[col] = 0.0  # Default momentum
                    elif col == 'rsi_14d':
                        filtered_df[col] = 50.0  # Default RSI
                    elif col == 'relative_volume':
                        filtered_df[col] = 1.0  # Default volume ratio
            
            self.aligned_data[ticker] = filtered_df[selected_cols]
        
        # Create a master date list from the common dates
        self.dates = common_dates
        
        # Initialize price matrix for quick lookup
        # Shape: (num_dates, num_stocks)
        self.price_matrix = np.zeros((len(self.dates), self.num_stocks))
        self.return_matrix = np.zeros((len(self.dates), self.num_stocks))
        
        for i, ticker in enumerate(self.tickers):
            price_data = self.aligned_data[ticker]['close'].values
            return_data = self.aligned_data[ticker]['daily_return'].values
            
            self.price_matrix[:len(price_data), i] = price_data
            self.return_matrix[:len(return_data), i] = return_data
        
        print(f"Environment prepared with {len(self.dates)} common dates.")

    def _prepare_data_most_recent(self):
        """
        Alternative data preparation that uses the most recent data for each stock
        without requiring perfect date overlap
        """
        print("Using alternative data preparation method with most recent data.")
        
        # Determine how many days of data we need minimum
        min_days_needed = self.max_steps + self.window_size
        
        # Process each stock to get the most recent data
        self.aligned_data = {}
        all_included_tickers = []
        
        for ticker, df in self.stock_data_dict.items():
            # Ensure the dataframe is sorted by date
            df = df.sort_values('date')
            
            # Check if we have enough data
            if len(df) < min_days_needed:
                print(f"Skipping {ticker} - insufficient data ({len(df)} rows)")
                continue
                
            # Take the most recent data
            recent_data = df.iloc[-min_days_needed:].copy()
            
            # Ensure all needed columns exist
            selected_cols = ['date', 'close', 'daily_return', 'volatility_20d', 
                             'momentum_20d', 'rsi_14d', 'relative_volume']
            
            for col in selected_cols:
                if col not in recent_data.columns:
                    print(f"Warning: Column {col} not found in data for {ticker}. Using default values.")
                    if col == 'close':
                        recent_data[col] = 100.0  # Default price
                    elif col == 'daily_return':
                        recent_data[col] = 0.0  # Default return
                    elif col == 'volatility_20d':
                        recent_data[col] = 0.2  # Default volatility
                    elif col == 'momentum_20d':
                        recent_data[col] = 0.0  # Default momentum
                    elif col == 'rsi_14d':
                        recent_data[col] = 50.0  # Default RSI
                    elif col == 'relative_volume':
                        recent_data[col] = 1.0  # Default volume ratio
            
            # Store the processed data
            self.aligned_data[ticker] = recent_data[selected_cols]
            all_included_tickers.append(ticker)
        
        # Update tickers list to only include those with sufficient data
        self.tickers = all_included_tickers
        self.num_stocks = len(self.tickers)
        
        if self.num_stocks == 0:
            raise ValueError("No stocks with sufficient data for the environment.")
        
        print(f"Using {self.num_stocks} stocks with sufficient data.")
        
        # Create a synthetic date list spanning the required days
        # (actual calendar dates don't matter for the simulation)
        self.dates = pd.date_range(start='2020-01-01', periods=min_days_needed)
        
        # Initialize matrices for quick lookup during simulation
        self.price_matrix = np.zeros((min_days_needed, self.num_stocks))
        self.return_matrix = np.zeros((min_days_needed, self.num_stocks))
        
        for i, ticker in enumerate(self.tickers):
            # Extract price and return data
            stock_data = self.aligned_data[ticker]
            self.price_matrix[:, i] = stock_data['close'].values
            self.return_matrix[:, i] = stock_data['daily_return'].values
        
        # Update action space to match new number of stocks
        self.action_space = spaces.Box(
            low=0, high=1, shape=(self.num_stocks,), dtype=np.float32
        )
        
        # Update observation space
        state_dim = (self.num_stocks * 5) + (self.num_stocks + 2)
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(state_dim,), dtype=np.float32
        )
        
        print(f"Environment prepared with {len(self.dates)} dates for {self.num_stocks} stocks.")
    
    def reset(self, seed=None, options=None):
        """Reset the environment to the beginning of an episode"""
        # Set seed if provided (Gymnasium API)
        if seed is not None:
            np.random.seed(seed)
            
        # Randomly select a starting point, ensuring enough history for the window
        # and enough future data for max_steps
        latest_start = len(self.dates) - self.max_steps
        earliest_start = self.window_size
        
        if latest_start <= earliest_start:
            self.start_idx = earliest_start
        else:
            self.start_idx = np.random.randint(earliest_start, latest_start)
        
        self.current_step = 0
        self.portfolio_value = self.initial_investment
        self.portfolio_history = [self.portfolio_value]
        self.risk_adjusted_returns = []
        
        # Equal allocation to start - ENSURE THIS IS ARRAY!
        self.allocations = np.ones(self.num_stocks) / self.num_stocks
        
        # Get initial observation
        return self._get_observation(), {}  # Return observation and empty info dict (Gymnasium API)
    
    def _get_observation(self):
        """Construct the current state observation"""
        idx = self.start_idx + self.current_step
        
        # Get window of historical data for each stock
        state_features = []
        
        for i, ticker in enumerate(self.tickers):
            stock_data = self.aligned_data[ticker]
            
            # Handle potential indexing issues
            if idx >= len(stock_data) or idx-self.window_size < 0:
                # Use default values if we're out of bounds
                normalized_price = 1.0
                volatility = 0.2
                momentum = 0.0
                rsi = 0.5
                volume_ratio = 1.0
            else:
                window_data = stock_data.iloc[idx-self.window_size:idx]
                
                # Normalize price to the last window's start price
                normalized_price = stock_data.iloc[idx]['close'] / window_data.iloc[0]['close']
                
                # Get other features - use the most recent values
                volatility = stock_data.iloc[idx]['volatility_20d']
                momentum = stock_data.iloc[idx]['momentum_20d']
                rsi = stock_data.iloc[idx]['rsi_14d'] / 100.0  # Normalize RSI to 0-1
                volume_ratio = stock_data.iloc[idx]['relative_volume']
            
            # Append features for this stock
            state_features.extend([normalized_price, volatility, momentum, rsi, volume_ratio])
        
        # Portfolio state
        # Current allocations - ENSURE THIS IS ITERABLE!
        if isinstance(self.allocations, (float, np.float64, np.float32)):
            print(f"Warning: Allocations is a single value: {self.allocations}. Converting to array.")
            # Reset to equal allocation if it's somehow a scalar
            self.allocations = np.ones(self.num_stocks) / self.num_stocks
            
        state_features.extend(self.allocations)
        
        # Portfolio return (over last window)
        if self.current_step >= self.window_size:
            portfolio_return = (self.portfolio_history[-1] / self.portfolio_history[-self.window_size]) - 1
        else:
            portfolio_return = 0.0
        
        # Portfolio volatility (standard deviation of daily returns)
        if len(self.risk_adjusted_returns) > 1:
            portfolio_volatility = np.std(self.risk_adjusted_returns[-self.window_size:]) * np.sqrt(252)
        else:
            portfolio_volatility = 0.0
        
        state_features.extend([portfolio_return, portfolio_volatility])
        
        return np.array(state_features, dtype=np.float32)
    
    def step(self, action):
        """
        Take an action (portfolio allocation) and move to the next step
        
        Args:
            action: numpy array of portfolio weights (will be normalized)
            
        Returns:
            observation: the current state observation
            reward: the reward for the action
            terminated: whether the episode is terminated
            truncated: whether the episode was truncated
            info: additional information
        """
        # Ensure action is valid (normalize to sum to 1)
        # Handle scalar action case
        if isinstance(action, (float, np.float64, np.float32)):
            print(f"Warning: Action is a single value: {action}. Converting to array.")
            action = np.ones(self.num_stocks) / self.num_stocks
            
        action = np.clip(action, 0, 1)
        if np.sum(action) > 0:
            action = action / np.sum(action)
        else:
            action = np.ones_like(action) / len(action)  # Equal allocation if all zeros
        
        # Store action as an array
        self.allocations = np.array(action).flatten()
        
        # Get return for current day based on previous allocation
        idx = self.start_idx + self.current_step
        if idx < len(self.return_matrix):
            daily_returns = self.return_matrix[idx]
        else:
            # If we're beyond our data (shouldn't happen), use zeros
            daily_returns = np.zeros(self.num_stocks)
        
        # Calculate portfolio return
        portfolio_return = np.sum(self.allocations * daily_returns)
        
        # Update portfolio value
        old_value = self.portfolio_value
        self.portfolio_value *= (1 + portfolio_return)
        self.portfolio_history.append(self.portfolio_value)
        self.risk_adjusted_returns.append(portfolio_return - self.risk_free_rate)
        
        # Calculate reward based on risk-adjusted return
        # Use Sharpe-like reward but for a single day
        if len(self.risk_adjusted_returns) > self.window_size:
            # Calculate Sharpe ratio over the window
            mean_return = np.mean(self.risk_adjusted_returns[-self.window_size:])
            std_return = np.std(self.risk_adjusted_returns[-self.window_size:]) 
            
            # Avoid division by zero
            if std_return > 0:
                sharpe = mean_return / std_return * np.sqrt(252)  # Annualize
            else:
                sharpe = 0.0
                
            # Penalize for high volatility or drawdowns
            try:
                volatility_penalty = np.sum(self.allocations * 
                                        [self.aligned_data[t].iloc[idx]['volatility_20d'] 
                                         for t in self.tickers])
            except:
                # If we have an indexing issue, use a default penalty
                volatility_penalty = 0.2
            
            # Calculate max drawdown over the window
            window_values = self.portfolio_history[-self.window_size:]
            peak = np.maximum.accumulate(window_values)
            drawdown = (window_values / peak) - 1
            max_drawdown = np.min(drawdown)
            
            # Calculate the final reward
            reward = sharpe - 0.5 * volatility_penalty - 2.0 * abs(max_drawdown)
        else:
            # Not enough history for good risk assessment
            reward = portfolio_return * 100  # Simple return-based reward initially
            
        # Move to the next step
        self.current_step += 1
        
        # Check if the episode is done
        terminated = self.current_step >= self.max_steps
        truncated = False  # We're not truncating episodes early
        
        # Get the new observation
        obs = self._get_observation()
        
        # Additional info
        info = {
            'portfolio_value': self.portfolio_value,
            'portfolio_return': portfolio_return,
            'sharpe_ratio': sharpe if 'sharpe' in locals() else 0,
            'allocations': self.allocations
        }
        
        return obs, reward, terminated, truncated, info  # Return observation, reward, terminated, truncated, info (Gymnasium API)
    
    def render(self):
        """Render the environment (optional)"""
        pass

    def get_portfolio_performance(self):
        """Get the performance metrics for the episode"""
        if len(self.portfolio_history) <= 1:
            return {
                'final_value': self.portfolio_value,
                'total_return': 0.0,
                'annualized_return': 0.0,
                'annualized_volatility': 0.0,
                'sharpe_ratio': 0.0,
                'max_drawdown': 0.0
            }
            
        returns = np.diff(self.portfolio_history) / self.portfolio_history[:-1]
        
        annualized_return = np.mean(returns) * 252
        annualized_vol = np.std(returns) * np.sqrt(252)
        sharpe = annualized_return / annualized_vol if annualized_vol > 0 else 0
        
        # Calculate max drawdown
        peak = np.maximum.accumulate(self.portfolio_history)
        drawdown = (self.portfolio_history / peak) - 1
        max_drawdown = np.min(drawdown)
        
        return {
            'final_value': self.portfolio_history[-1],
            'total_return': (self.portfolio_history[-1] / self.initial_investment) - 1,
            'annualized_return': annualized_return,
            'annualized_volatility': annualized_vol,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_drawdown
        }

The StockRiskEnv class already includes its own data preparation steps (in _prepare_data() and _prepare_data_most_recent() methods), but it's designed to work with data that has already been partially preprocessed by your prepare_stock_data() function.
Here's the workflow:

First,  prepare_stock_data() function processes the raw CSV data into a dictionary of DataFrames with all the necessary features (like daily returns, volatility, momentum, RSI, etc.)
Then, the StockRiskEnv class takes this pre-processed data and does additional environment-specific preparations like:

Finding common date ranges
Aligning the data across stocks
Organizing it into formats needed for the RL environment

This two-step approach is good practice because:

It separates concerns - the initial data processing is separate from the environment logic
It makes your code more modular and reusable
It allows you to use the preprocessed data for other analyses beyond just the RL environment

In [104]:
# Custom callback for logging training progress

class StockRiskCallback(BaseCallback):
    def __init__(self, eval_env, verbose=0, eval_freq=1000):
        super(StockRiskCallback, self).__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.portfolio_values = []
        self.returns = []
        self.sharpe_ratios = []
        self.max_drawdowns = []
        self.best_sharpe = -np.inf
        self.best_model_path = None
        
    def _on_step(self):
        # Evaluate agent periodically
        if self.n_calls % self.eval_freq == 0:
            try:
                # Run a full episode for evaluation
                obs = self.eval_env.reset()
                done = False
                
                while not done:
                    action, _ = self.model.predict(obs, deterministic=True)
                    obs, _, done, info = self.eval_env.step(action)
                    done = done[0]  # Extract scalar value from array
                
                # Get portfolio performance metrics
                performance = self.eval_env.get_attr('get_portfolio_performance')[0]()
                
                self.portfolio_values.append(performance['final_value'])
                self.returns.append(performance['total_return'])
                self.sharpe_ratios.append(performance['sharpe_ratio'])
                self.max_drawdowns.append(performance['max_drawdown'])
                
                # Log metrics
                self.logger.record('eval/portfolio_value', performance['final_value'])
                self.logger.record('eval/total_return', performance['total_return'])
                self.logger.record('eval/sharpe_ratio', performance['sharpe_ratio'])
                self.logger.record('eval/max_drawdown', performance['max_drawdown'])
                
                # Save best model based on Sharpe ratio
                if performance['sharpe_ratio'] > self.best_sharpe:
                    self.best_sharpe = performance['sharpe_ratio']
                    if self.best_model_path:
                        self.model.save(self.best_model_path)
                        print(f"New best model saved with Sharpe ratio: {self.best_sharpe:.4f}")
                
                print(f"Step {self.n_calls}: Portfolio value: ${performance['final_value']:.2f}, "
                     f"Return: {performance['total_return']*100:.2f}%, "
                     f"Sharpe: {performance['sharpe_ratio']:.2f}, "
                     f"Max Drawdown: {performance['max_drawdown']*100:.2f}%")
            except Exception as e:
                print(f"Error in evaluation: {str(e)}")
                import traceback
                traceback.print_exc()
                
        return True

'''class StockRiskCallback(BaseCallback):
    def __init__(self, eval_env, verbose=0, eval_freq=1000):
        super(StockRiskCallback, self).__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.portfolio_values = []
        self.returns = []
        self.sharpe_ratios = []
        self.max_drawdowns = []
        self.best_sharpe = -np.inf
        self.best_model_path = None
        
    def _on_step(self):
        # Evaluate agent periodically
        if self.n_calls % self.eval_freq == 0:
            # Run a full episode for evaluation
            obs, _ = self.eval_env.reset()
            done = False
            terminated = False
            truncated = False
            
            while not done:
                action, _ = self.model.predict(obs, deterministic=True)
                obs, _, terminated, truncated, _ = self.eval_env.step(action)
                done = terminated or truncated
            
            # Get portfolio performance metrics
            performance = self.eval_env.env.get_portfolio_performance()
            
            self.portfolio_values.append(performance['final_value'])
            self.returns.append(performance['total_return'])
            self.sharpe_ratios.append(performance['sharpe_ratio'])
            self.max_drawdowns.append(performance['max_drawdown'])
            
            # Log metrics
            self.logger.record('eval/portfolio_value', performance['final_value'])
            self.logger.record('eval/total_return', performance['total_return'])
            self.logger.record('eval/sharpe_ratio', performance['sharpe_ratio'])
            self.logger.record('eval/max_drawdown', performance['max_drawdown'])
            
            # Save best model based on Sharpe ratio
            if performance['sharpe_ratio'] > self.best_sharpe:
                self.best_sharpe = performance['sharpe_ratio']
                if self.best_model_path:
                    self.model.save(self.best_model_path)
                    print(f"New best model saved with Sharpe ratio: {self.best_sharpe:.4f}")
            
            print(f"Step {self.n_calls}: Portfolio value: ${performance['final_value']:.2f}, "
                  f"Return: {performance['total_return']*100:.2f}%, "
                  f"Sharpe: {performance['sharpe_ratio']:.2f}, "
                  f"Max Drawdown: {performance['max_drawdown']*100:.2f}%")
            
        return True

    def get_metrics(self):
        return {
            'portfolio_values': self.portfolio_values,
            'returns': self.returns,
            'sharpe_ratios': self.sharpe_ratios,
            'max_drawdowns': self.max_drawdowns,
            'best_sharpe': self.best_sharpe
        }'''
def train_sac_model(stock_data_dict, total_timesteps=20000, 
                   save_path="models/sac_stock_risk", eval_freq=10000,
                   log_dir="logs/"):
    import psutil

    # Check available memory
    mem_available = psutil.virtual_memory().available / (1024**3)  # Convert to GB
    print(f"Available Memory: {mem_available:.2f} GB")

    if mem_available < 4:
        print("Warning: Low memory detected! Reducing training parameters.")
        buffer_size = 25000  # Reduce memory footprint
        batch_size = 32  # Smaller batch size
    else:
        buffer_size = 50000  # Optimized for better performance
        batch_size = 64  

    # Create the environments
    env = StockRiskEnv(stock_data_dict)
    eval_env = StockRiskEnv(stock_data_dict)
    
    # Get state dimensions safely
    state_dim = env.observation_space.shape
    if isinstance(state_dim, tuple):
        state_dim = state_dim[0]
    
    action_dim = env.action_space.shape[0]
    print(f"State dimension: {state_dim}, Action dimension: {action_dim}")
    
    # Create vectorized environments
    env = DummyVecEnv([lambda: env])
    eval_env = DummyVecEnv([lambda: eval_env])
    
    # Initialize action noise for exploration
    action_noise = NormalActionNoise(
        mean=np.zeros(action_dim), 
        sigma=0.1 * np.ones(action_dim)
    )
    
    # Set up logging without tensorboard
    os.makedirs(log_dir, exist_ok=True)
    logger = configure(log_dir, ["stdout", "csv"])  
    
    # Create the model
    model = SAC(
        policy="MlpPolicy",
        env=env,
        learning_rate=3e-4,
        buffer_size=buffer_size,  # Adjusted
        batch_size=batch_size,  # Adjusted
        gamma=0.99,
        tau=0.005,
        policy_kwargs=dict(net_arch=[256, 256]),
        action_noise=action_noise,
        verbose=1,
        tensorboard_log=None
    )
    
    # Set up custom callback
    callback = StockRiskCallback(eval_env=eval_env, eval_freq=eval_freq)
    callback.best_model_path = save_path
    
    # Train the model
    model.set_logger(logger)
    model.learn(total_timesteps=total_timesteps, callback=callback)
    
    # Save the final model
    model.save(save_path)
    print(f"Model saved to {save_path}")
    
    return model, callback


'''def train_sac_model(stock_data_dict, total_timesteps=20000, 
                   save_path="models/sac_stock_risk", eval_freq=10000,
                   log_dir="logs/"):
    # Create the environments
    env = StockRiskEnv(stock_data_dict)
    eval_env = StockRiskEnv(stock_data_dict)
    
    # Get dimensions before vectorizing
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    print(f"State dimension: {state_dim}, Action dimension: {action_dim}")
    
    # Create vectorized environments
    env = DummyVecEnv([lambda: env])
    eval_env = DummyVecEnv([lambda: eval_env])
    
    # Initialize action noise for exploration
    action_noise = NormalActionNoise(
        mean=np.zeros(action_dim), 
        sigma=0.1 * np.ones(action_dim)
    )
    
    # Set up logging without tensorboard
    os.makedirs(log_dir, exist_ok=True)
    logger = configure(log_dir, ["stdout", "csv"])  
    
    # Create the model
    model = SAC(
        policy="MlpPolicy",
        env=env,
        learning_rate=3e-4,
        buffer_size=100000,
        batch_size=256,
        gamma=0.99,
        tau=0.005,
        policy_kwargs=dict(net_arch=[256, 256]),
        action_noise=action_noise,
        verbose=1,
        tensorboard_log=None  # Disable tensorboard logging
    )
    
    # Set up custom callback
    callback = StockRiskCallback(eval_env=eval_env, eval_freq=eval_freq)
    callback.best_model_path = save_path
    
    # Train the model
    model.set_logger(logger)
    model.learn(total_timesteps=total_timesteps, callback=callback)
    
    # Save the final model
    model.save(save_path)
    print(f"Model saved to {save_path}")
    
    return model, callback'''


def evaluate_sac_model(model, stock_data_dict, num_episodes=10):
    """
    Evaluate a trained SAC model on stock risk assessment
    
    Args:
        model: Trained SAC model
        stock_data_dict: Dictionary of stock DataFrames
        num_episodes: Number of episodes to evaluate
        
    Returns:
        results: Dictionary with evaluation metrics
    """
    # Create environment for evaluation
    eval_env = StockRiskEnv(stock_data_dict)
    eval_env = DummyVecEnv([lambda: eval_env])
    
    # Storage for metrics
    portfolio_values = []
    returns = []
    sharpe_ratios = []
    max_drawdowns = []
    allocation_history = []
    
    # Evaluate over multiple episodes
    for episode in range(num_episodes):
        obs, _ = eval_env.reset()
        done = False
        terminated = False
        truncated = False
        step_allocations = []
        
        while not done:
            # Use deterministic actions for evaluation
            action, _ = model.predict(obs, deterministic=True)
            step_allocations.append(action[0])  # Store allocation at each step
            
            obs, _, terminated, truncated, _ = eval_env.step(action)
            done = terminated or truncated
        
        # Get metrics for the episode
        performance = eval_env.get_attr('get_portfolio_performance')[0]()
        
        portfolio_values.append(performance['final_value'])
        returns.append(performance['total_return'])
        sharpe_ratios.append(performance['sharpe_ratio'])
        max_drawdowns.append(performance['max_drawdown'])
        allocation_history.append(step_allocations)
        
        print(f"Episode {episode+1}: Portfolio value: ${performance['final_value']:.2f}, "
              f"Return: {performance['total_return']*100:.2f}%, "
              f"Sharpe: {performance['sharpe_ratio']:.2f}, "
              f"Max Drawdown: {performance['max_drawdown']*100:.2f}%")
    
    # Calculate average metrics
    avg_return = np.mean(returns)
    avg_sharpe = np.mean(sharpe_ratios)
    avg_drawdown = np.mean(max_drawdowns)
    
    print(f"\nAverage over {num_episodes} episodes:")
    print(f"  Return: {avg_return*100:.2f}%")
    print(f"  Sharpe Ratio: {avg_sharpe:.2f}")
    print(f"  Max Drawdown: {avg_drawdown*100:.2f}%")
    
    return {
        'portfolio_values': portfolio_values,
        'returns': returns,
        'sharpe_ratios': sharpe_ratios,
        'max_drawdowns': max_drawdowns,
        'allocation_history': allocation_history,
        'avg_return': avg_return,
        'avg_sharpe': avg_sharpe,
        'avg_drawdown': avg_drawdown
    }

In [ ]:

def classify_risk_with_sac(stock_data_dict, sac_model, risk_df=None, window_size=60):
    """
    Classify stock risk using a trained SAC model
    
    Args:
        stock_data_dict: Dictionary of stock DataFrames
        sac_model: Trained SAC model
        risk_df: Optional DataFrame with traditional risk metrics (for comparison)
        window_size: Window size for risk assessment
        
    Returns:
        risk_classification_df: DataFrame with risk classifications
    """
    try:
        print("Starting SAC risk classification...")
        
        # Create a test environment with the same stocks used for training
        env = StockRiskEnv(stock_data_dict, window_size=window_size)
        
        # Get tickers from environment
        tickers = env.tickers
        
        # Create vectorized environment for stable-baselines
        vec_env = DummyVecEnv([lambda: env])
        
        # Get initial observation - handle both API versions
        obs = vec_env.reset()
        print(f"Observation shape for prediction: {obs.shape}")
        
        # Get allocation action from model
        action, _ = sac_model.predict(obs, deterministic=True)
        print(f"Predicted action shape: {action.shape}")
        
        # Store allocations (flatten if needed)
        if len(action.shape) > 1:
            allocations = action[0]
        else:
            allocations = action
        
        print(f"Using allocations with shape: {allocations.shape} for {len(tickers)} tickers")
        
        # Verify dimensions match
        if len(allocations) != len(tickers):
            print(f"Warning: allocation length ({len(allocations)}) doesn't match tickers ({len(tickers)})")
            # Truncate to shorter length
            min_len = min(len(allocations), len(tickers))
            allocations = allocations[:min_len]
            tickers = tickers[:min_len]
        
        # Create risk classification DataFrame
        risk_classification = pd.DataFrame({
            'ticker': tickers,
            'sac_allocation': allocations
        })
        
        # Calculate SAC risk score based on inverse of allocation
        if np.sum(allocations) > 0:  # Check for valid allocations
            # Normalize allocations to sum to 1
            normalized_allocations = allocations / np.sum(allocations)
            
            # Higher allocation = lower risk, so invert and normalize to 0-1
            inverse_allocations = 1 - normalized_allocations / np.max(normalized_allocations)
            
            # Scale to range 0-1
            if np.max(inverse_allocations) > np.min(inverse_allocations):
                sac_risk_scores = (inverse_allocations - np.min(inverse_allocations)) / (np.max(inverse_allocations) - np.min(inverse_allocations))
            else:
                sac_risk_scores = np.zeros_like(inverse_allocations)
        else:
            # Default to equal risk if no valid allocations
            sac_risk_scores = np.ones(len(tickers)) * 0.5
        
        risk_classification['sac_risk_score'] = sac_risk_scores
        
        # Classify into risk categories
        try:
            risk_classification['sac_risk_category'] = pd.qcut(
                risk_classification['sac_risk_score'], 
                q=3, 
                labels=['Low', 'Medium', 'High']
            )
        except ValueError:
            # If we can't create 3 categories (e.g., too many identical values)
            # Use manual classification
            print("Warning: Could not create equal-size risk categories. Using manual thresholds.")
            risk_classification['sac_risk_category'] = pd.cut(
                risk_classification['sac_risk_score'],
                bins=[0, 0.33, 0.67, 1],
                labels=['Low', 'Medium', 'High'],
                include_lowest=True
            )
        
        # If traditional risk metrics are provided, merge for comparison
        if risk_df is not None:
            # Make sure both DataFrames have the tickers in the same format
            risk_df['ticker'] = risk_df['ticker'].astype(str)
            risk_classification['ticker'] = risk_classification['ticker'].astype(str)
            
            # Merge with traditional risk metrics
            risk_classification = pd.merge(
                risk_classification, 
                risk_df[['ticker', 'risk_score', 'risk_category', 'volatility', 'max_drawdown', 'sharpe_ratio']], 
                on='ticker', 
                how='left',
                suffixes=('', '_traditional')
            )
            
            # Rename columns for clarity
            risk_classification = risk_classification.rename(columns={
                'risk_score': 'traditional_risk_score',
                'risk_category': 'traditional_risk_category'
            })
        
        print("SAC risk classification completed successfully!")
        return risk_classification
        
    except Exception as e:
        print(f"Error in SAC risk classification: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

def select_training_stocks(stock_dfs, n=50, selection_method='volume'):
    """
    Select a subset of stocks for training based on specific criteria
    
    Args:
        stock_dfs: Dictionary of stock DataFrames
        n: Number of stocks to select
        selection_method: Method for selection ('volume', 'random')
        
    Returns:
        Dictionary with selected stocks
    """
    selected_stocks = {}
    ranking = []
    
    # Calculate selection metric for each stock
    for ticker, df in stock_dfs.items():
        if selection_method == 'volume':
            # Select based on average trading volume
            metric = df['volume'].mean() if 'volume' in df.columns else 0
        elif selection_method == 'random':
            # Random selection
            metric = np.random.random()
        else:
            # Default to volume
            metric = df['volume'].mean() if 'volume' in df.columns else 0
            
        ranking.append((ticker, metric))
    
    # Sort by metric and take top n
    ranking.sort(key=lambda x: x[1], reverse=True)
    selected_tickers = [item[0] for item in ranking[:n]]
    
    # Create dictionary with selected stocks
    for ticker in selected_tickers:
        selected_stocks[ticker] = stock_dfs[ticker]
        
    print(f"Selected {len(selected_stocks)} stocks for training")
    return selected_stocks


def compare_risk_classifications(risk_df):
    """
    Compare SAC and traditional risk classifications
    
    Args:
        risk_df: DataFrame with both SAC and traditional risk classifications
        
    Returns:
        comparison_metrics: Dictionary with comparison metrics
    """
    # Check if we have both classification methods
    if 'sac_risk_category' not in risk_df.columns or 'traditional_risk_category' not in risk_df.columns:
        print("Error: Risk DataFrame does not contain both SAC and traditional risk classifications")
        return None
    
    # Calculate agreement percentage
    agreement = np.mean(risk_df['sac_risk_category'] == risk_df['traditional_risk_category'])
    agreement_pct = agreement * 100
    
    # Create contingency table
    contingency = pd.crosstab(
        risk_df['traditional_risk_category'], 
        risk_df['sac_risk_category'],
        normalize='index'
    ) * 100
    
    # Calculate correlation between risk scores
    correlation = risk_df['sac_risk_score'].corr(risk_df['traditional_risk_score'])
    
    # Print comparison results
    print(f"Risk Classification Comparison:")
    print(f"Agreement: {agreement_pct:.2f}%")
    print(f"Correlation between risk scores: {correlation:.4f}")
    print("\nContingency Table (% of traditional categories):")
    print(contingency)
    
    # Return metrics
    return {
        'agreement': agreement,
        'correlation': correlation,
        'contingency_table': contingency
    }


def visualize_risk_comparison(risk_df):
    """
    Create visualizations comparing SAC and traditional risk classifications
    
    Args:
        risk_df: DataFrame with both SAC and traditional risk classifications
    """
    # Check if we have both classification methods
    if 'sac_risk_category' not in risk_df.columns or 'traditional_risk_category' not in risk_df.columns:
        print("Error: Risk DataFrame does not contain both SAC and traditional risk classifications")
        return None
    
    # Set up the figure
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Scatter plot of risk scores
    ax1 = axes[0, 0]
    sns.scatterplot(
        x='traditional_risk_score', 
        y='sac_risk_score', 
        hue='traditional_risk_category',
        palette='viridis',
        size='volatility',
        sizes=(20, 200),
        alpha=0.7,
        data=risk_df,
        ax=ax1
    )
    ax1.set_title('Comparison of Risk Scores')
    ax1.set_xlabel('Traditional Risk Score')
    ax1.set_ylabel('SAC Risk Score')
    ax1.plot([0, 1], [0, 1], 'r--', alpha=0.5)  # Diagonal line for reference
    
    # 2. Heatmap of contingency table
    ax2 = axes[0, 1]
    contingency = pd.crosstab(
        risk_df['traditional_risk_category'], 
        risk_df['sac_risk_category']
    )
    sns.heatmap(
        contingency, 
        annot=True, 
        fmt='d', 
        cmap='Blues', 
        ax=ax2
    )
    ax2.set_title('Confusion Matrix: Traditional vs. SAC Risk Categories')
    ax2.set_xlabel('SAC Risk Category')
    ax2.set_ylabel('Traditional Risk Category')
    
    # 3. Risk score distributions
    ax3 = axes[1, 0]
    sns.kdeplot(
        data=risk_df, 
        x='traditional_risk_score', 
        label='Traditional',
        fill=True,
        alpha=0.5,
        ax=ax3
    )
    sns.kdeplot(
        data=risk_df, 
        x='sac_risk_score', 
        label='SAC',
        fill=True,
        alpha=0.5,
        ax=ax3
    )
    ax3.set_title('Distribution of Risk Scores')
    ax3.set_xlabel('Risk Score')
    ax3.set_ylabel('Density')
    ax3.legend()
    
    # 4. Risk allocation by volatility
    ax4 = axes[1, 1]
    sns.scatterplot(
        x='volatility', 
        y='sac_allocation',
        hue='sharpe_ratio',
        palette='coolwarm',
        size='traditional_risk_score',
        sizes=(20, 200),
        alpha=0.7,
        data=risk_df,
        ax=ax4
    )
    ax4.set_title('SAC Allocation vs. Volatility')
    ax4.set_xlabel('Volatility (Annualized)')
    ax4.set_ylabel('SAC Portfolio Allocation')
    
    plt.tight_layout()
    
    return fig

In [105]:
def main():
    print("=" * 80)
    print("STOCK RISK ASSESSMENT PIPELINE WITH SOFT ACTOR-CRITIC (SAC)")
    print("=" * 80)
    
    # Step 1: Data Preparation
    print("\n1. Preparing stock data...")
    try:
        stock_dfs = prepare_stock_data(file_path)
        print(f"Processed {len(stock_dfs)} stocks.")
        
        # Debug: Check the first few stocks
        for i, (ticker, df) in enumerate(list(stock_dfs.items())[:3]):
            print(f"Sample stock {ticker}: {len(df)} rows, columns: {list(df.columns)}")
            if 'date' in df.columns:
                print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
            if i == 0:
                print(f"  First few rows:")
                print(df.head(3))
    except Exception as e:
        print(f"Error in data preparation: {str(e)}")
        return
    
    # Step 2: Calculate Traditional Risk Metrics
    print("\n2. Calculating traditional risk metrics...")
    try:
        risk_df = calculate_risk_metrics(stock_dfs)
        print(f"Generated risk metrics for {len(risk_df)} stocks.")
        
        # Print distribution of traditional risk categories
        print("\nTraditional Risk Distribution:")
        print(risk_df['risk_category'].value_counts())
        
        # Save traditional risk metrics
        risk_df.to_csv(os.path.join(RESULTS_DIR, "traditional_risk_metrics.csv"), index=False)
    except Exception as e:
        print(f"Error in risk metric calculation: {str(e)}")
        return
    
    # Step 3: Train SAC Model
    print("\n3. Training SAC model...")
    
    model_path = os.path.join(MODEL_DIR, "sac_stock_risk")
    
    try:
        # Check if model exists already to avoid retraining
        if os.path.exists(model_path + ".zip"):
            print(f"Loading existing model from {model_path}.zip")
            from stable_baselines3 import SAC
            model = SAC.load(model_path)
            callback = None
        else:
            print("Training new SAC model...")
            model, callback = train_sac_model(
                stock_data_dict=stock_dfs,
                total_timesteps=20000, 
                save_path=model_path,
                eval_freq=5000,
                log_dir=LOG_DIR
            )
            
            # Plot training metrics if we have them
            if callback:
                metrics = callback.get_metrics()
                
                plt.figure(figsize=(15, 10))
                
                # Plot Sharpe Ratio
                plt.subplot(2, 2, 1)
                plt.plot(metrics['sharpe_ratios'])
                plt.title('Sharpe Ratio During Training')
                plt.xlabel('Evaluation Step')
                plt.ylabel('Sharpe Ratio')
                
                # Plot Portfolio Return
                plt.subplot(2, 2, 2)
                plt.plot([r*100 for r in metrics['returns']])
                plt.title('Portfolio Return During Training')
                plt.xlabel('Evaluation Step')
                plt.ylabel('Return (%)')
                
                # Plot Max Drawdown
                plt.subplot(2, 2, 3)
                plt.plot([md*100 for md in metrics['max_drawdowns']])
                plt.title('Max Drawdown During Training')
                plt.xlabel('Evaluation Step')
                plt.ylabel('Max Drawdown (%)')
                
                # Plot Portfolio Value
                plt.subplot(2, 2, 4)
                plt.plot(metrics['portfolio_values'])
                plt.title('Portfolio Value During Training')
                plt.xlabel('Evaluation Step')
                plt.ylabel('Portfolio Value ($)')
                
                plt.tight_layout()
                plt.savefig(os.path.join(RESULTS_DIR, "training_metrics.png"))
    except Exception as e:
        print(f"Error in model training: {str(e)}")
        return
    
    # Step 4: Evaluate SAC Model
    print("\n4. Evaluating SAC model performance...")
    try:
        eval_results = evaluate_sac_model(model, stock_dfs, num_episodes=5)
        
        # Save evaluation results
        eval_df = pd.DataFrame({
            'episode': range(1, len(eval_results['returns']) + 1),
            'return': eval_results['returns'],
            'sharpe_ratio': eval_results['sharpe_ratios'],
            'max_drawdown': eval_results['max_drawdowns']
        })
        eval_df.to_csv(os.path.join(RESULTS_DIR, "evaluation_results.csv"), index=False)
        
        # Plot evaluation results
        plt.figure(figsize=(15, 10))
        
        # Plot Returns
        plt.subplot(2, 2, 1)
        plt.bar(range(len(eval_results['returns'])), [r*100 for r in eval_results['returns']])
        plt.axhline(y=eval_results['avg_return']*100, color='r', linestyle='--', label=f'Avg: {eval_results["avg_return"]*100:.2f}%')
        plt.title('Returns by Episode')
        plt.xlabel('Episode')
        plt.ylabel('Return (%)')
        plt.legend()
        
        # Plot Sharpe Ratios
        plt.subplot(2, 2, 2)
        plt.bar(range(len(eval_results['sharpe_ratios'])), eval_results['sharpe_ratios'])
        plt.axhline(y=eval_results['avg_sharpe'], color='r', linestyle='--', label=f'Avg: {eval_results["avg_sharpe"]:.2f}')
        plt.title('Sharpe Ratios by Episode')
        plt.xlabel('Episode')
        plt.ylabel('Sharpe Ratio')
        plt.legend()
        
        # Plot Max Drawdowns
        plt.subplot(2, 2, 3)
        plt.bar(range(len(eval_results['max_drawdowns'])), [md*100 for md in eval_results['max_drawdowns']])
        plt.axhline(y=eval_results['avg_drawdown']*100, color='r', linestyle='--', label=f'Avg: {eval_results["avg_drawdown"]*100:.2f}%')
        plt.title('Max Drawdowns by Episode')
        plt.xlabel('Episode')
        plt.ylabel('Max Drawdown (%)')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, "evaluation_results.png"))
    except Exception as e:
        print(f"Error in model evaluation: {str(e)}")
        # Continue anyway, as we can still do risk classification
    
    


    print("\n5. Classifying stock risk with SAC model...")
    try:
        # Try with a reduced set of stocks to match training conditions
        top_stocks = select_training_stocks(stock_dfs, n=50, selection_method='volume')
        risk_classification_df = classify_risk_with_sac(top_stocks, model, risk_df)
    
        # Save raw risk classification results
        if risk_classification_df is not None:
            risk_classification_df.to_csv(os.path.join(RESULTS_DIR, "sac_risk_classification.csv"), index=False)
        
        # Print distribution of SAC risk categories
            print("\nSAC Risk Distribution:")
            print(risk_classification_df['sac_risk_category'].value_counts())
        else:
            print("Could not generate risk classification")
    except Exception as e:
        print(f"Error in risk classification: {str(e)}")
        import traceback
        traceback.print_exc()

    '''# Step 5: Risk Classification with SAC
    print("\n5. Classifying stock risk with SAC model...")
    try:
        risk_classification_df = classify_risk_with_sac(stock_dfs, model, risk_df)
        
        # Save raw risk classification results
        risk_classification_df.to_csv(os.path.join(RESULTS_DIR, "sac_risk_classification.csv"), index=False)
        
        # Print distribution of SAC risk categories
        print("\nSAC Risk Distribution:")
        print(risk_classification_df['sac_risk_category'].value_counts())
    except Exception as e:
        print(f"Error in risk classification: {str(e)}")
        return'''
    
    # Step 6: Compare Risk Classification Methods
    print("\n6. Comparing risk classification methods...")
    try:
        comparison_metrics = compare_risk_classifications(risk_classification_df)
        
        # Save comparison metrics
        if comparison_metrics:
            metrics_df = pd.DataFrame({
                'metric': ['agreement', 'correlation'],
                'value': [comparison_metrics['agreement'], comparison_metrics['correlation']]
            })
            metrics_df.to_csv(os.path.join(RESULTS_DIR, "comparison_metrics.csv"), index=False)
            
            # Save contingency table
            comparison_metrics['contingency_table'].to_csv(
                os.path.join(RESULTS_DIR, "contingency_table.csv")
            )
        
        # Visualize the comparison
        try:
            comparison_fig = visualize_risk_comparison(risk_classification_df)
            if comparison_fig:
                comparison_fig.savefig(os.path.join(RESULTS_DIR, "risk_comparison.png"))
        except Exception as viz_error:
            print(f"Warning: Could not create visualization: {str(viz_error)}")
    except Exception as e:
        print(f"Error in risk comparison: {str(e)}")
        # Continue anyway to save the final results
    
    # Step 7: Save Final Results
    print("\n7. Saving final risk analysis results...")
    try:
        save_risk_analysis(risk_classification_df, os.path.join(RESULTS_DIR, "risk_analysis_results.csv"))
    except Exception as e:
        print(f"Error saving final results: {str(e)}")
    
    print("\nProcess completed successfully!")
    print(f"Results saved to {RESULTS_DIR} directory")
    
    return {
        'risk_df': risk_df,
        'model': model,
        'risk_classification_df': risk_classification_df,
        'eval_results': eval_results if 'eval_results' in locals() else None,
        'comparison_metrics': comparison_metrics if 'comparison_metrics' in locals() else None
    }

In [106]:
# Run the main pipeline
if __name__ == "__main__":
    results = main()
else:
    # For Jupyter notebook execution
    results = main()

STOCK RISK ASSESSMENT PIPELINE WITH SOFT ACTOR-CRITIC (SAC)

1. Preparing stock data...
Original data shape: (4066965, 8)
Skipping ACAA.TO - insufficient data (121 rows)
Skipping ADIV.TO - insufficient data (184 rows)
Skipping AGG.TO - insufficient data (51 rows)
Skipping AIGO.TO - insufficient data (156 rows)
Skipping AMAX.TO - insufficient data (225 rows)
Skipping AMHE.TO - insufficient data (89 rows)
Skipping AMZH.TO - insufficient data (90 rows)
Skipping AW.TO - insufficient data (50 rows)
Skipping BCHT.TO - insufficient data (47 rows)
Skipping CAPG.TO - insufficient data (46 rows)
Skipping CAPI.TO - insufficient data (46 rows)
Skipping CAPM.TO - insufficient data (46 rows)
Skipping CAPW.TO - insufficient data (46 rows)
Skipping CBL.TO - insufficient data (140 rows)
Skipping CIAI.TO - insufficient data (163 rows)
Skipping CMOM.TO - insufficient data (241 rows)
Skipping CNDX.TO - insufficient data (156 rows)
Skipping CNV.TO - insufficient data (140 rows)
Skipping COPR.TO - insuffici

c:\Users\nikhi\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


Generated risk metrics for 1462 stocks.

Traditional Risk Distribution:
risk_category
Low       488
High      488
Medium    486
Name: count, dtype: int64

3. Training SAC model...
Loading existing model from models\sac_stock_risk.zip
Error in model training: Unable to allocate 3.05 GiB for an array with shape (100000, 1, 8174) and data type float32


### - **HYPERPARAMETER TUNING AND FINE-TUNING** 

In [ ]:
def select_training_stocks(stock_dfs, n=50, selection_method='volume'):
    """
    Select a subset of stocks for training based on specific criteria
    
    Args:
        stock_dfs: Dictionary of stock DataFrames
        n: Number of stocks to select
        selection_method: Method for selection ('volume', 'liquidity', 'market_cap', 'random')
        
    Returns:
        Dictionary with selected stocks
    """
    selected_stocks = {}
    ranking = []
    
    # Calculate selection metric for each stock
    for ticker, df in stock_dfs.items():
        if selection_method == 'volume':
            # Select based on average trading volume
            metric = df['volume'].mean() if 'volume' in df.columns else 0
        elif selection_method == 'liquidity':
            # Select based on average turnover
            metric = (df['volume'] * df['close']).mean() if 'volume' in df.columns and 'close' in df.columns else 0
        elif selection_method == 'random':
            # Random selection
            metric = np.random.random()
        else:
            # Default to volume
            metric = df['volume'].mean() if 'volume' in df.columns else 0
            
        ranking.append((ticker, metric))
    
    # Sort by metric and take top n
    ranking.sort(key=lambda x: x[1], reverse=True)
    selected_tickers = [item[0] for item in ranking[:n]]
    
    # Create dictionary with selected stocks
    for ticker in selected_tickers:
        selected_stocks[ticker] = stock_dfs[ticker]
        
    print(f"Selected {len(selected_stocks)} stocks for training")
    return selected_stocks

In [ ]:
def optimized_train_sac(stock_data_dict, total_timesteps=50000, 
                        save_path="models/optimized_sac", eval_freq=1000,
                        log_dir="logs/", batch_size=256, learning_rate=3e-4,
                        gamma=0.99, buffer_size=100000):
    """
    Optimized SAC training with better hyperparameters and configuration
    """
    # Create environments
    env = StockRiskEnv(stock_data_dict, max_steps=100, window_size=20)
    eval_env = StockRiskEnv(stock_data_dict, max_steps=100, window_size=20)
    
    # Get dimensions before vectorizing
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    print(f"State dimension: {state_dim}, Action dimension: {action_dim}")
    
    # Create vectorized environments
    env = DummyVecEnv([lambda: env])
    eval_env = DummyVecEnv([lambda: eval_env])
    
    # Initialize action noise for exploration with adaptive scaling
    action_noise = NormalActionNoise(
        mean=np.zeros(action_dim), 
        sigma=0.2 * np.ones(action_dim)  # Increased exploration
    )
    
    # Set up logging
    os.makedirs(log_dir, exist_ok=True)
    logger = configure(log_dir, ["stdout", "csv"])
    
    # Create the model with optimized hyperparameters
    model = SAC(
        policy="MlpPolicy",
        env=env,
        learning_rate=learning_rate,
        buffer_size=buffer_size,
        batch_size=batch_size,
        gamma=gamma,
        tau=0.005,
        ent_coef='auto',  # Automatic entropy adjustment
        policy_kwargs=dict(
            net_arch=[256, 256],  # Can be adjusted based on complexity
            activation_fn=torch.nn.ReLU,
            use_sde=False  # Try with and without state-dependent exploration
        ),
        action_noise=action_noise,
        verbose=1,
        tensorboard_log=None
    )
    
    # Set up custom callback
    callback = StockRiskCallback(eval_env=eval_env, eval_freq=eval_freq)
    callback.best_model_path = save_path
    
    # Train the model
    model.set_logger(logger)
    model.learn(total_timesteps=total_timesteps, callback=callback)
    
    # Save the final model
    model.save(save_path)
    print(f"Model saved to {save_path}")
    
    return model, callback

In [ ]:
def create_env_with_custom_reward(stock_data_dict, reward_type='sharpe'):
    """
    Create an environment with a custom reward function
    
    Args:
        stock_data_dict: Dictionary of stock DataFrames
        reward_type: Type of reward function ('sharpe', 'sortino', 'return', 'risk_adjusted')
        
    Returns:
        Environment with custom reward function
    """
    # Create a custom class that inherits from StockRiskEnv
    class CustomRewardEnv(StockRiskEnv):
        def __init__(self, stock_data_dict, reward_type='sharpe', **kwargs):
            super().__init__(stock_data_dict, **kwargs)
            self.reward_type = reward_type
            print(f"Using {reward_type} reward function")
        
        def step(self, action):
            # Ensure action is valid (normalize to sum to 1)
            if isinstance(action, (float, np.float64, np.float32)):
                action = np.ones(self.num_stocks) / self.num_stocks
                
            action = np.clip(action, 0, 1)
            if np.sum(action) > 0:
                action = action / np.sum(action)
            else:
                action = np.ones_like(action) / len(action)
            
            # Store action as array
            self.allocations = np.array(action).flatten()
            
            # Get return for current day
            idx = self.start_idx + self.current_step
            if idx < len(self.return_matrix):
                daily_returns = self.return_matrix[idx]
            else:
                daily_returns = np.zeros(self.num_stocks)
            
            # Calculate portfolio return
            portfolio_return = np.sum(self.allocations * daily_returns)
            
            # Update portfolio value
            self.portfolio_value *= (1 + portfolio_return)
            self.portfolio_history.append(self.portfolio_value)
            self.risk_adjusted_returns.append(portfolio_return - self.risk_free_rate)
            
            # Calculate custom reward based on specified type
            if len(self.risk_adjusted_returns) > self.window_size:
                # Get window of returns
                window_returns = self.risk_adjusted_returns[-self.window_size:]
                mean_return = np.mean(window_returns)
                std_return = np.std(window_returns)
                
                if self.reward_type == 'sharpe':
                    # Sharpe ratio (default)
                    reward = mean_return / std_return * np.sqrt(252) if std_return > 0 else 0
                
                elif self.reward_type == 'sortino':
                    # Sortino ratio (penalizes only downside deviation)
                    downside_returns = [r for r in window_returns if r < 0]
                    downside_deviation = np.std(downside_returns) if downside_returns else 0
                    reward = mean_return / downside_deviation * np.sqrt(252) if downside_deviation > 0 else 0
                
                elif self.reward_type == 'return':
                    # Pure return-based reward
                    reward = mean_return * 252  # Annualized return
                
                elif self.reward_type == 'risk_adjusted':
                    # Custom risk-adjusted reward with drawdown penalty
                    window_values = self.portfolio_history[-self.window_size:]
                    peak = np.maximum.accumulate(window_values)
                    drawdown = (window_values / peak) - 1
                    max_drawdown = np.min(drawdown)
                    
                    # Penalize drawdowns heavily
                    reward = mean_return * 252 - 3.0 * std_return * np.sqrt(252) - 5.0 * abs(max_drawdown)
                else:
                    # Default to Sharpe
                    reward = mean_return / std_return * np.sqrt(252) if std_return > 0 else 0
            else:
                # Not enough history
                reward = portfolio_return * 100
            
            # Move to next step
            self.current_step += 1
            
            # Check if episode is done
            terminated = self.current_step >= self.max_steps
            truncated = False
            
            # Get new observation
            obs = self._get_observation()
            
            # Additional info
            info = {
                'portfolio_value': self.portfolio_value,
                'portfolio_return': portfolio_return,
                'reward_type': self.reward_type,
                'allocations': self.allocations
            }
            
            return obs, reward, terminated, truncated, info
    
    # Create and return the environment
    return CustomRewardEnv(stock_data_dict, reward_type=reward_type)

In [ ]:
def run_training_experiments(stock_dfs, base_path="models/experiments"):
    """
    Run a series of training experiments with different configurations
    
    Args:
        stock_dfs: Dictionary of all stock DataFrames
        base_path: Base path for saving models and results
        
    Returns:
        Dictionary with results from all experiments
    """
    results = {}
    
    # Create directories
    os.makedirs(base_path, exist_ok=True)
    
    # --- EXPERIMENT 1: Stock Selection Methods ---
    for selection_method in ['volume', 'random']:
        # Select stocks
        selected_stocks = select_training_stocks(stock_dfs, n=50, selection_method=selection_method)
        
        # Train model
        model_path = f"{base_path}/sac_{selection_method}_stocks"
        try:
            model, callback = optimized_train_sac(
                selected_stocks,
                total_timesteps=20000,
                save_path=model_path,
                eval_freq=1000
            )
            
            # Store results
            results[f"selection_{selection_method}"] = {
                'model': model,
                'metrics': callback.get_metrics() if hasattr(callback, 'get_metrics') else None
            }
            
        except Exception as e:
            print(f"Error in {selection_method} stock experiment: {str(e)}")
    
    # --- EXPERIMENT 2: Different Reward Functions ---
    volume_stocks = select_training_stocks(stock_dfs, n=30, selection_method='volume')
    
    for reward_type in ['sharpe', 'sortino', 'risk_adjusted']:
        # Create environment with custom reward
        env = create_env_with_custom_reward(volume_stocks, reward_type=reward_type)
        eval_env = create_env_with_custom_reward(volume_stocks, reward_type=reward_type)
        
        # Vectorize
        vec_env = DummyVecEnv([lambda: env])
        vec_eval_env = DummyVecEnv([lambda: eval_env])
        
        # Train model
        model_path = f"{base_path}/sac_{reward_type}_reward"
        try:
            # Get dimensions
            state_dim = env.observation_space.shape[0]
            action_dim = env.action_space.shape[0]
            
            # Create model
            model = SAC(
                policy="MlpPolicy",
                env=vec_env,
                learning_rate=3e-4,
                buffer_size=50000,
                batch_size=256,
                verbose=1
            )
            
            # Set up callback
            callback = StockRiskCallback(eval_env=vec_eval_env, eval_freq=1000)
            callback.best_model_path = model_path
            
            # Train
            model.learn(total_timesteps=15000, callback=callback)
            model.save(model_path)
            
            # Store results
            results[f"reward_{reward_type}"] = {
                'model': model,
                'metrics': callback.get_metrics() if hasattr(callback, 'get_metrics') else None
            }
            
        except Exception as e:
            print(f"Error in {reward_type} reward experiment: {str(e)}")
    
    return results

In [ ]:
def analyze_experiments(results, stock_dfs, save_path="results/experiments"):
    """
    Analyze and compare results from different training experiments
    
    Args:
        results: Dictionary with experiment results
        stock_dfs: Dictionary of stock DataFrames
        save_path: Path to save analysis results
        
    Returns:
        DataFrame with comparative metrics
    """
    os.makedirs(save_path, exist_ok=True)
    
    # Create comparison dataframe
    comparison = []
    
    for exp_name, exp_data in results.items():
        if exp_data.get('metrics') is None:
            continue
            
        metrics = exp_data['metrics']
        
        # Calculate key performance indicators
        avg_sharpe = np.mean(metrics['sharpe_ratios']) if 'sharpe_ratios' in metrics else 0
        avg_return = np.mean(metrics['returns']) if 'returns' in metrics else 0
        avg_drawdown = np.mean(metrics['max_drawdowns']) if 'max_drawdowns' in metrics else 0
        best_sharpe = metrics.get('best_sharpe', 0)
        
        # Add to comparison
        comparison.append({
            'experiment': exp_name,
            'avg_sharpe': avg_sharpe,
            'avg_return': avg_return * 100,  # Percentage
            'avg_drawdown': avg_drawdown * 100,  # Percentage
            'best_sharpe': best_sharpe
        })
    
    # Convert to DataFrame
    comparison_df = pd.DataFrame(comparison)
    
    # Save to CSV
    comparison_df.to_csv(f"{save_path}/experiment_comparison.csv", index=False)
    
    return comparison_df

In [ ]:
# Select a small subset of stocks for initial experimentation
training_stocks = select_training_stocks(stock_dfs, n=30, selection_method='volume')

# Train a model with optimized settings
model, callback = optimized_train_sac(
    training_stocks,
    total_timesteps=10000,  # Keep small for initial testing
    save_path="models/optimized_sac",
    eval_freq=500
)

# Evaluate the model
if callback is not None:
    metrics = callback.get_metrics()
    print(f"Best Sharpe Ratio: {metrics.get('best_sharpe', 0):.4f}")
    print(f"Average Return: {np.mean(metrics.get('returns', [0])) * 100:.2f}%")

Selected 30 stocks for training
Preparing environment with 30 stocks
Stock CNQ.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock XIU.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock SU.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock MFC.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock TD.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock K.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock CVE.TO has 3744 rows with date range: 2010-01-28 00:00:00 to 2024-12-30 00:00:00
Stock ABX.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock ENB.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock BTO.TO has 4175 rows with date range: 2008-05-09 00:00:00 to 2024-12-30 00:00:00
Stock RY.TO has 5212 rows with date range: 2004-03-26 00:00:00 to

AttributeError: 'StockRiskCallback' object has no attribute 'get_metrics'

### **DEBUGGING AND LOGGING**

In [ ]:
# Run this cell before calling debug_train
stock_dfs = prepare_stock_data(file_path)

Original data shape: (4066965, 8)
Skipping ACAA.TO - insufficient data (121 rows)
Skipping ADIV.TO - insufficient data (184 rows)
Skipping AGG.TO - insufficient data (51 rows)
Skipping AIGO.TO - insufficient data (156 rows)
Skipping AMAX.TO - insufficient data (225 rows)
Skipping AMHE.TO - insufficient data (89 rows)
Skipping AMZH.TO - insufficient data (90 rows)
Skipping AW.TO - insufficient data (50 rows)
Skipping BCHT.TO - insufficient data (47 rows)
Skipping CAPG.TO - insufficient data (46 rows)
Skipping CAPI.TO - insufficient data (46 rows)
Skipping CAPM.TO - insufficient data (46 rows)
Skipping CAPW.TO - insufficient data (46 rows)
Skipping CBL.TO - insufficient data (140 rows)
Skipping CIAI.TO - insufficient data (163 rows)
Skipping CMOM.TO - insufficient data (241 rows)
Skipping CNDX.TO - insufficient data (156 rows)
Skipping CNV.TO - insufficient data (140 rows)
Skipping COPR.TO - insufficient data (96 rows)
Skipping CORE.TO - insufficient data (92 rows)
Skipping CSBI.TO - ins

In [ ]:
def debug_train(stock_data_dict, max_stocks=10, max_steps=50):
    """
    Simplified training function to debug the tuple index error
    """
    print("Starting simplified debug training...")
    
    # Take a small subset of stocks to reduce complexity
    test_stocks = {}
    for i, (ticker, df) in enumerate(stock_data_dict.items()):
        test_stocks[ticker] = df
        if i >= max_stocks - 1:  # Use only max_stocks
            break
    
    print(f"Using {len(test_stocks)} stocks for debugging")
    
    # Create a simplified environment
    env = StockRiskEnv(test_stocks, max_steps=max_steps, window_size=10)
    
    # Directly test the environment
    print("Testing environment reset and step...")
    obs, _ = env.reset()
    print(f"Reset successful. Observation shape: {obs.shape}")
    
    action = env.action_space.sample()
    print(f"Sampled action shape: {action.shape}")
    
    try:
        next_obs, reward, terminated, truncated, info = env.step(action)
        print(f"Step successful! Reward: {reward}")
    except Exception as e:
        print(f"Error during environment step: {str(e)}")
        import traceback
        traceback.print_exc()
        return None
    
    # Now try with vectorized environment
    print("\nTesting with DummyVecEnv...")
    vec_env = DummyVecEnv([lambda: StockRiskEnv(test_stocks, max_steps=max_steps, window_size=10)])
    
    try:
        # Test reset
        obs = vec_env.reset()
        print(f"Vectorized reset successful! Observation shape: {obs.shape}")
        
        # Test step
        action = vec_env.action_space.sample()
        print(f"Vectorized action shape: {action.shape}")
        
        next_obs, reward, done, info = vec_env.step(action)
        print(f"Vectorized step successful! Reward shape: {reward.shape}")
    except Exception as e:
        print(f"Error with vectorized environment: {str(e)}")
        import traceback
        traceback.print_exc()
        return None
    
    # If we get here, the environment is working correctly
    # Now try simplified training with SAC
    print("\nTrying simplified SAC training...")
    
    try:
        model = SAC(
            policy="MlpPolicy",
            env=vec_env,
            learning_rate=3e-4,
            batch_size=64,
            policy_kwargs=dict(net_arch=[64, 64]),  # Smaller network
            verbose=1
        )
        
        # Train for just a few steps
        model.learn(total_timesteps=100)  # Start with a very small number
        print("Training successful!")
        return model
    except Exception as e:
        print(f"Error during training: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
def test_env_shapes():
    """Test environment shapes to ensure proper vectorization"""
    # Create a tiny dataset for testing
    test_stocks = {}
    for i, (ticker, df) in enumerate(stock_dfs.items()):
        test_stocks[ticker] = df
        if i >= 2:  # Just need 3 stocks
            break
    
    # Create base environment
    base_env = StockRiskEnv(test_stocks, max_steps=20, window_size=5)
    print(f"Base env observation space: {base_env.observation_space}")
    print(f"Base env observation shape: {base_env.observation_space.shape}")
    print(f"Base env action space: {base_env.action_space}")
    print(f"Base env action shape: {base_env.action_space.shape}")
    
    # Create vectorized environment
    vec_env = DummyVecEnv([lambda: StockRiskEnv(test_stocks, max_steps=20, window_size=5)])
    print(f"Vec env observation space: {vec_env.observation_space}")
    print(f"Vec env observation shape: {vec_env.observation_space.shape}")
    print(f"Vec env action space: {vec_env.action_space}")
    print(f"Vec env action shape: {vec_env.action_space.shape}")
    
    return "Shape test complete"

# First make sure you've loaded your stock data
if 'stock_dfs' not in locals() and 'stock_dfs' not in globals():
    print("Loading stock data...")
    stock_dfs = prepare_stock_data(DATA_PATH)
    
# Now run the test
result = test_env_shapes()
print(result)

Preparing environment with 3 stocks
Stock AAB.TO has 4777 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock AAUC.TO has 266 rows with date range: 2023-12-07 00:00:00 to 2024-12-30 00:00:00
Stock AAV.TO has 3746 rows with date range: 2010-01-26 00:00:00 to 2024-12-30 00:00:00
Found 250 common dates across all stocks.
Environment prepared with 250 common dates.
Base env observation space: Box(-inf, inf, (20,), float32)
Base env observation shape: (20,)
Base env action space: Box(0.0, 1.0, (3,), float32)
Base env action shape: (3,)
Preparing environment with 3 stocks
Stock AAB.TO has 4777 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock AAUC.TO has 266 rows with date range: 2023-12-07 00:00:00 to 2024-12-30 00:00:00
Stock AAV.TO has 3746 rows with date range: 2010-01-26 00:00:00 to 2024-12-30 00:00:00
Found 250 common dates across all stocks.
Environment prepared with 250 common dates.
Vec env observation space: Box(-inf, inf, (20,), float32)
Vec

In [ ]:
# Step 3: Train SAC Model
print("\n3. Training SAC model...")
# Default: 500,000 timesteps, adjust as needed for your computational resources
model_path = os.path.join(MODEL_DIR, "sac_stock_risk")

try:
    # Check if model exists already to avoid retraining
    if os.path.exists(model_path + ".zip"):
        print(f"Loading existing model from {model_path}.zip")
        from stable_baselines3 import SAC
        model = SAC.load(model_path)
    else:
        print("Debugging training first...")
        debug_model = debug_train(stock_dfs, max_stocks=10, max_steps=50)
        
        if debug_model is not None:
            print("Debug training successful, proceeding with full training...")
            # Now proceed with actual training using fewer stocks initially
            small_stock_dict = {}
            for i, (ticker, df) in enumerate(stock_dfs.items()):
                small_stock_dict[ticker] = df
                if i >= 50:  # Use 50 stocks initially
                    break
                    
            # Train with 50 stocks first
            model, callback = train_sac_model(
                stock_data_dict=small_stock_dict,
                total_timesteps=20000,  # Short training for testing
                save_path=model_path,
                eval_freq=1000,
                log_dir=LOG_DIR
            )
        else:
            print("Debug training failed, cannot proceed with full training")
        print("Exiting training process due to debug training failure.")
except Exception as e:
    print(f"Error in model training: {str(e)}")
    import traceback
    traceback.print_exc()


3. Training SAC model...
Debugging training first...
Starting simplified debug training...
Using 10 stocks for debugging
Preparing environment with 10 stocks
Stock AAB.TO has 4777 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock AAUC.TO has 266 rows with date range: 2023-12-07 00:00:00 to 2024-12-30 00:00:00
Stock AAV.TO has 3746 rows with date range: 2010-01-26 00:00:00 to 2024-12-30 00:00:00
Stock ABX.TO has 5212 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock AC.TO has 3640 rows with date range: 2010-06-24 00:00:00 to 2024-12-30 00:00:00
Stock ACB.TO has 2005 rows with date range: 2017-01-03 00:00:00 to 2024-12-30 00:00:00
Stock ACD.TO has 4172 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock ACQ.TO has 4062 rows with date range: 2008-10-21 00:00:00 to 2024-12-30 00:00:00
Stock ACX.TO has 5211 rows with date range: 2004-03-26 00:00:00 to 2024-12-30 00:00:00
Stock ADCO.TO has 1319 rows with date range: 2019-08-23 00:

Traceback (most recent call last):
  File "C:\Users\nikhi\AppData\Local\Temp\ipykernel_7692\4234039532.py", line 26, in <module>
    model, callback = train_sac_model(
                      ^^^^^^^^^^^^^^^^
  File "C:\Users\nikhi\AppData\Local\Temp\ipykernel_7692\3668770798.py", line 93, in train_sac_model
    state_dim = eval_env.observation_space.shape[1]  # For vec envs, first dim is num_envs
                ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^
IndexError: tuple index out of range
